# 让 AI 访谈更可控的方法

目标：让 AI 访谈更像真人对话，同时保持可控。

核心观点：**通常不需要先做模型微调**。先从提示工程、工作流、评估迭代入手，成本低、见效快。微调放在后期，用来固化风格或领域习惯。

## 0. 当前系统是怎么工作的

1. 用户配置模型提供商、API Key、模型。
2. 输入研究目标。
3. 系统把研究目标包装成系统提示词，让 AI 生成第一个问题。
4. 每次用户回答后，把完整对话历史再发给 AI，让它生成下一个问题。
5. 用户点击结束，系统把完整对话发给 AI，让它生成结构化报告。

可控性不足的地方：AI 自由发挥较多，没有明确的阶段控制，追问策略单一。

## 1. 提示工程（Prompt Engineering）

最轻量、最先尝试的方法。

### 1.1 系统提示词更具体
- 定义角色：经验丰富的定性研究主持人。
- 定义阶段：开场 → 背景 → 核心探索 → 深度追问 → 收尾。
- 定义语气：对话式、尊重、不评价。
- 定义约束：一次只问一个问题，不总结、不分析。

### 1.2 Few-shot 示例
在提示词里加入 3-5 段高质量真人访谈片段，AI 会模仿这种节奏和追问方式。

### 1.3 结构化输出
要求 AI 每轮返回 JSON，包含：
- `question`：下一个问题
- `stage`：当前阶段
- `reason`：为什么问这个问题

这样后端可以校验、干预，甚至拒绝不符合阶段的问题。

In [ ]:
# 示例：更具体的系统提示词
system_prompt = """
你是一位经验丰富的定性研究访谈主持人。
研究目标：{goal}

访谈阶段：
1. 开场：建立信任，说明目的，问一个轻松的开放问题。
2. 背景：了解受访者的基本情况和使用场景。
3. 核心探索：围绕研究目标深入挖掘。
4. 深度追问：对关键回答追问动机、感受、具体例子。
5. 收尾：总结确认，感谢受访者。

规则：
- 一次只问一个问题。
- 语气自然、对话式，避免像问卷。
- 追问时先说一小句对回答的理解，再问下一个问题。
- 如果受访者回答简短， gently 请他举个例子。
- 不输出分析、总结、bullet list。
- 输出必须是 JSON：{{"question": "...", "stage": "...", "reason": "..."}}
"""
print(system_prompt[:200])

## 2. 状态机 / 工作流（State Machine）

把访谈拆成明确的阶段，每轮根据上下文判断当前阶段，再调用对应提示词。

优点：
- 可控性强，每个阶段有固定策略。
- 可以插入口袋问题（fallback questions）。
- 容易评估和调试。

实现方式：
- 硬编码规则：轮数、关键词、情绪检测。
- 让 AI 自己判断阶段：每轮先调一个 cheap 模型做阶段分类，再调主模型生成问题。
- 混合：关键节点用规则，其余交给 AI。

In [ ]:
# 示例：简单的阶段判断函数
def infer_stage(messages, goal):
    n = len(messages)
    if n <= 1:
        return 'opening'
    if n <= 3:
        return 'background'
    if n <= 6:
        return 'core_exploration'
    if any('为什么' in m['text'] or '原因' in m['text'] for m in messages[-2:] if m['role'] == 'user'):
        return 'deep_probing'
    return 'closing'

messages = [
    {'role': 'assistant', 'text': '你好，能简单介绍一下自己吗？'},
    {'role': 'user', 'text': '我是一名产品经理，平时做用户调研。'},
]
print(infer_stage(messages, '了解用户研究方法'))

## 3. 检索增强（RAG）

把访谈提纲、过往优秀访谈、领域知识做成向量库。每次生成问题前，先检索相关参考，再让 AI 参照生成。

适用场景：
- 公司有一套访谈方法论（如 Jobs-to-be-Done、JTBD）。
- 想复用某次特别成功的访谈风格。
- 让 AI 自动引用特定概念或框架。

不需要自己训练模型，只需要好的数据和向量检索。

## 4. 函数调用 / 工具（Function Calling / Tools）

让 AI 在特定节点调用工具，而不是纯文本自由发挥。

可以定义的工具：
- `ask_question(question, stage)`：正常提问。
- `probe(reason, target)`：追问某个点。
- `summarize_and_confirm()`：阶段小结并请用户确认。
- `end_interview()`：结束访谈。

后端收到函数调用后，可以执行、校验、或改写。这样 AI 的输出就被限制在一组预定义动作里。

In [ ]:
# 示例：工具定义（伪代码，OpenAI / Anthropic 格式类似）
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'ask_question',
            'description': '向受访者提出下一个问题',
            'parameters': {
                'type': 'object',
                'properties': {
                    'question': {'type': 'string', 'description': '问题内容'},
                    'stage': {'type': 'string', 'enum': ['opening', 'background', 'core_exploration', 'deep_probing', 'closing']},
                    'reason': {'type': 'string', 'description': '为什么问这个问题'}
                },
                'required': ['question', 'stage', 'reason']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'end_interview',
            'description': '结束访谈并感谢受访者',
            'parameters': {'type': 'object', 'properties': {}}
        }
    }
]
print(len(tools))

## 5. 评估与迭代（Eval Loop）

没有评估，就不知道哪种方法更好。

评估维度：
- 问题是否自然、不机械？
- 追问是否切中要点？
- 是否跑题或过早下结论？
- 受访者回答长度是否合适？
- 最终报告是否覆盖研究目标？

方法：
- 人工抽检 10-20 段对话。
- 用另一个 AI 模型当评委，按维度打分。
- 记录 bad case，改提示词或工作流。
- 做 A/B 测试：对比不同提示词版本。

In [ ]:
# 示例：简单的评估打分表
rubric = {
    'naturalness': '问题是否像真人对话，不生硬',
    'relevance': '问题是否紧扣研究目标',
    'probing': '是否基于用户回答做了有效追问',
    'single_question': '一次是否只问一个问题',
    'no_bias': '是否没有引导性或偏见性语言',
}
for k, v in rubric.items():
    print(f'{k}: {v}')

## 6. 模型微调（Fine-tuning）

什么时候才考虑微调？
- 提示词和工作流已经做到 80 分，但某些风格、领域术语、追问节奏始终调不好。
- 积累了大量高质量真人访谈数据，想复制某位优秀主持人的风格。
- 需要模型稳定地遵守特定方法论（如 JTBD、五问法）。

微调方法：

| 方法 | 说明 | 成本 | 适用场景 |
|---|---|---|---|
| **SFT（监督微调）** | 用高质量问答对训练模型 | 中等 | 模仿特定访谈风格、固定格式 |
| **LoRA / QLoRA** | 只训练少量参数，效率更高 | 低 | 快速实验、资源有限 |
| **RLHF / DPO** | 用人类偏好数据优化 | 高 | 让模型学会更自然的对话偏好 |
| **全量微调** | 训练整个模型 | 很高 | 追求最佳效果，通常没必要 |

注意：微调需要成对的训练数据，比如（对话上下文 → 优秀主持人下一句话）。数据质量比数据量更重要。

## 7. 推荐落地路线

1. **先优化系统提示词**：加入阶段定义、few-shot 示例、结构化输出。
2. **引入状态机**：用规则 + 小模型做阶段判断，控制访谈节奏。
3. **加入 RAG**：检索访谈提纲和优秀案例。
4. **用函数调用限制输出**：把提问、追问、结束变成可调用的工具。
5. **建立评估循环**：收集 bad case，打分，迭代。
6. **必要时微调**：当提示词 + 工作流的天花板明显，且有足够数据时，再考虑 LoRA 或 SFT。

对于 ow-text 当前这个项目，建议先做第 1-2 步，就能明显提升可控性。

## 8. Mizzen 功能映射

Mizzen 提示的预访谈设计功能，与本文方法的对应关系：

| Mizzen 功能 | 实现方法 | 优先级 |
|---|---|---|
| 创建访谈提纲与问题 | 提示工程 + RAG 检索优秀提纲 | 高 |
| 定义目标受众与场景 | 状态机分支（不同受众走不同流程） | 中 |
| 设计访谈流程与结构 | 状态机（阶段定义 + 转换规则） | 高 |
| 优化问题措辞 | 评估循环 + few-shot 示例 + A/B 测试 | 中 |
| 生成个性化访谈指南 | 结构化输出 + 按项目/受访者模板替换 | 中 |

## 9. Worklog

记录已完成的探索与决策。

- `2026-07-26 11:04:55 CST` 创建 `notes.ipynb`，整理让 AI 访谈更可控的方法体系。
- `2026-07-26 11:04:55 CST` 补充 Mizzen 预访谈设计功能映射。
- `2026-07-26 11:04:55 CST` 建立 Worklog 与 TDL 区域，便于后续迭代跟踪。
- `2026-07-26 11:29:15 CST` 调研外部开源项目、数据集与框架资源，补充进 notes。
- `2026-07-26 11:37:12 CST` 实现访谈状态机、结构化提示词与评估循环，并更新前端和测试。

## 10. TDL（待办清单）

- [x] `2026-07-26` 调研外部开源项目、数据集与框架资源，补充进 notes。
- [x] `2026-07-26` **实现访谈状态机**：开场 → 背景 → 核心探索 → 深度追问 → 收尾，按阶段切换提示词。
- [x] `2026-07-26` **升级系统提示词**：加入 few-shot 示例，要求 AI 返回结构化 JSON（question / stage / reason）。
- [x] `2026-07-26` **建立评估循环**：定义打分维度，新增 `/evaluate` 接口，前端可一键评估对话质量。
- [x] `2026-07-26` 更新前端，展示当前阶段、追问理由、评估结果。
- [ ] `2026-07-26` 收集并整理高质量访谈提纲/案例，准备 RAG 知识库（后续）。
- [ ] `2026-07-26` 调研并记录 LoRA / SFT 微调方案，作为后续备选。

## 11. 外部资源与开源项目

下面是在互联网上找到的、与 AI 访谈和用户研究相关的开源项目、数据集、框架与模型。可以复用或参考。

### 11.1 开源访谈 / 用户研究工具

| 项目 | 说明 | 链接 |
|---|---|---|
| **cookiy-ai/user-research-skill** | AI Skill 套件，端到端用户研究：AI 访谈、合成用户、定量问卷 | https://github.com/cookiy-ai/user-research-skill |
| **cookiy-ai/cookiy-cli** | 命令行版用户研究工具，支持从终端跑 AI 访谈 | https://github.com/cookiy-ai/cookiy-cli |
| **CyannSHI/ai-interview-kit** | 把用户研究方法学（6 种方法论）嵌入提示词，可规模化电话访谈 | https://github.com/CyannSHI/ai-interview-kit |
| **GaoKab/user-research-skill** | 客户研究 skill：规划研究、跑访谈、生成有证据支撑的报告 | https://github.com/GaoKab/user-research-skill |
| **chuxin-wenxiang/virtual_user_skill** | 基于 5.4 万+真实匿名用户研究场景生成 AI 虚拟用户，用于模拟访谈 | https://github.com/chuxin-wenxiang/virtual_user_skill |
| **nagoli/user-research-helper** | Human-in-the-loop 助手，做访谈转录和洞察分析 | https://github.com/nagoli/user-research-helper |
| **zzzlip/langgraph-AI-interview-agent** | 基于 LangGraph 的多智能体招聘/面试辅助系统 | https://github.com/zzzlip/langgraph-AI-interview-agent |

### 11.2 对话 / 访谈数据集

| 数据集 | 说明 | 链接 |
|---|---|---|
| **MediaSum** | 大规模媒体访谈数据集，用于对话摘要 | https://github.com/zcgzcgzcg1/MediaSum |
| **Motivational Interviewing Dataset** | 约 2K 段对话，倾听者话语带有动机访谈标签 | https://github.com/anuradha1992/Motivational-Interviewing-Dataset |
| **KMI** | 韩文动机访谈数据集（NAACL 2025），用于心理治疗对话 | https://github.com/hjkim811/KMI |
| **persona-dojo** | 从语音、对话、访谈中生成微调数据集，训练 AI persona | https://github.com/Ctrl-Alt-Cr8/persona-dojo |

这些数据集适合：
- 做 few-shot 示例；
- 训练/微调追问策略；
- 评估 AI 访谈自然度。

### 11.3 可复用的 Agent 框架

| 框架 | 适用场景 | 链接 |
|---|---|---|
| **LangGraph** | 多步骤、有状态、可分支的访谈流程 | https://github.com/langchain-ai/langgraph |
| **LangChain** | 模型调用、提示词模板、RAG、工具集成 | https://github.com/langchain-ai/langchain |
| **OpenAI Agents SDK** | 简单 Agent、工具调用、 handoff | https://github.com/openai/openai-agents-python |
| **AutoGen** | 多 Agent 协作，适合模拟多角色访谈 | https://github.com/microsoft/autogen |

### 11.4 可选用的基础模型

访谈任务不需要专用模型，通用对话模型即可。推荐：

| 模型 | 特点 |
|---|---|
| OpenAI GPT-4o / GPT-4o-mini | 多语言、指令遵循强，适合快速原型 |
| Anthropic Claude 3.5 Sonnet / 4 | 长上下文、对话自然、适合深度追问 |
| DeepSeek-V3 / DeepSeek-R1 | 中文好、成本低，推理版适合复杂报告生成 |
| Qwen2.5 / Qwen3 | 中文优秀，开源可本地部署 |
| Llama 3.1 / 3.2 / 4 | 开源生态丰富，可配合 LoRA 微调 |

### 11.5 微调训练资源

| 资源 | 说明 | 链接 |
|---|---|---|
| **Hugging Face TRL** | 训练 Transformer 语言模型（SFT、DPO、PPO） | https://github.com/huggingface/trl |
| **unsloth** | 快速微调 Llama/Qwen，显存占用低 | https://github.com/unslothai/unsloth |
| **LLaMA-Factory** | 一站式 LoRA/QLoRA/全量微调界面 | https://github.com/hiyouga/LLaMA-Factory |
| **Axolotl** | YAML 配置训练 LoRA，适合批量实验 | https://github.com/axolotl-ai-cloud/axolotl |

建议：先不要微调，先用提示词 + 工作流 + 评估把天花板摸到；有稳定数据后再用 LoRA 固化追问风格。